---------------------------

---

# 📊 Ders Notu: Büyük Veri Dünyasında Pandas vs. Spark

## 1. Temel Farklar ve Ölçeklenebilirlik
Veri işlemede iki ana yaklaşım vardır: **Dikey** ve **Yatay** ölçekleme.

* **Pandas (Dikey Ölçekleme):** Tek bir makine (single node) üzerinde çalışır. Performansı o makinenin **RAM kapasitesiyle** sınırlıdır.
* **Spark (Yatay Ölçekleme):** Birden fazla makineyi (cluster) birleştirerek çalışır. Performansı artırmak için daha pahalı bir makine almak yerine, sisteme **yeni makineler eklemek** yeterlidir.

| Özellik | Pandas | Spark |
| :--- | :--- | :--- |
| **Mimari** | Tek Makine | Dağıtılmış (Distributed) |
| **Hafıza Yönetimi** | Tek bir RAM | Kümedeki tüm RAM'lerin toplamı |
| **İşleme Kapasitesi** | Megabayt - Birkaç GB | Terabayt - Petabayt |
| **Kullanım Kolaylığı** | Çok yüksek (Sezgisel) | Orta (Planlama gerektirir) |



---

## 2. Büyük Veri Çözümleri: Hadoop vs. Spark
Büyük veri dendiğinde akla gelen iki dev isim arasındaki farklar şöyledir:

* **Hadoop:** Hesaplama sırasında ara verileri **diske yazar**. Bu yüzden daha az RAM gerektirir ancak daha **yavaştır**.
* **Spark:** Hesaplamaları doğrudan **bellekte (In-Memory)** yapar. Hadoop'tan çok daha **performanslıdır**.
* **PySpark:** Spark'ın Scala (JVM) ile yazılmış ana gövdesine Python ile erişmemizi sağlayan kütüphanedir.

---

## 3. Değerlendirme Stratejileri (Evaluation)
Verinin ne zaman ve nasıl işleneceğine dair iki farklı felsefe:

### A. İstekli (Eager) Değerlendirme - *Pandas Yaklaşımı*
Kodun her satırı çalıştırıldığı anda hesaplama yapılır.
* **Artısı:** Hata ayıklamak (debugging) çok kolaydır, sonuç anında görülür.
* **Eksisi:** Büyük veri setlerinde her adımda gereksiz kopyalar oluşturup sistemi yorar.

### B. Tembel (Lazy) Değerlendirme - *Spark Yaklaşımı*
İşlemler biriktirilir (istiflenir), ancak bir sonuç (Action) istenene kadar hesaplama başlatılmaz.
* **Artısı:** Spark, arka planda tüm adımları görür ve en verimli yolu (Execution Plan) çizer.
* **Eksisi:** Bir hata olduğunda, hatanın hangi adımda gerçekleştiğini bulmak daha zordur.



---

## 4. Pratik Limitler ve Seçim Kriterleri
Bir veri setinde ne zaman araç değiştirmeli?

* **0 - 5 GB:** Kesinlikle **Pandas**. Hızlı ve pratiktir.
* **5 - 100 GB:** **Pandas (Chunking)** veya **Dask**. Tek makinede sınırları zorlama aşaması.
* **100 GB + :** **Spark**. Dağıtılmış mimari artık bir zorunluluktur.

---

## 5. Teknik Zorluklar
* **Hata Mesajları:** PySpark hataları 3 katmanlıdır: Python İstisnası ➔ Çeviri Katmanı ➔ Scala/JVM Yığın İzi.
* **Stratejik Planlama:** Hangi komutların (örneğin `.show()`, `.count()`, `.collect()`) hesaplamayı tetikleyeceğini bilmek, performansı yönetmek için kritiktir.

---

## PYSPARK

#### Dask

In [1]:
import pandas as pd

In [2]:
import pandas as pd
import numpy as np   
import random

In [3]:
length=100000000    #``100 million rows, 2 columns olusturduk.
data={'a': (random.randint(0, 100) for _ in range(length)),
      'b': (random.randint(0, 100) for _ in range(length))}

In [4]:
df=pd.DataFrame(data)  #100 million rows, 2 columns olusturduk.
df.head(5) 

,a,b
0,34,16
1,33,55
2,27,22
3,52,45
4,2,57


In [5]:
df.std()

a    29.155368
b    29.153650
dtype: float64

ayni islemleri ____DASK____ ile yapalim

In [6]:
import dask.dataframe as dd 

In [7]:
ddf=dd.from_pandas(df, npartitions=3)  #Dask dataframe'e cevirdik. Parcalara bolduk.
ddf

,a,b
npartitions=3,,
0,int64,int64
33333334,...,...
66666667,...,...
99999999,...,...


In [11]:
ddf.std()  #Ciktilari vermez. Ciktilari goruntulemek icin compute() fonksiyonunu kullanmamiz gerekiyor.

Dask Series Structure:
npartitions=1
a    float64
b        ...
Dask Name: sqrt, 4 expressions
Expr=MapPartitions(sqrt)

In [12]:
ddf.std().compute() #ciktilari goruntulemek icin compute() fonksiyonunu kullanmamiz gerekiyor.

a    29.155368
b    29.153650
dtype: float64

In [15]:
result=ddf.a.sum() - ddf.b.std() #Dask dataframe'de islemi yaparken sonucunu goruntulemek icin compute() fonksiyonunu kullanmamiz gerekiyor.
result

<dask_expr.expr.Scalar: expr=df['a'].sum() - MapPartitions(sqrt), dtype=float64>

In [17]:
result=ddf.a.sum() - ddf.b.std().compute() #a kolonunun toplamini hesapladik ve compute() ile sonucu goruntuleyebildik.
result.compute() #Sonucu goruntulemek icin compute() fonksiyonunu kullanmamiz gerekiyor

np.float64(5000039179.846351)

In [19]:
result.dask

{('sub-49d68af32090f730a22647f787b4d8d4',
  0): <Task ('sub-49d68af32090f730a22647f787b4d8d4', 0) sub(...)>,
 ('sum-tree-63b7d2be8a88cc6da701c211bade5bbb',
  0): (<function dask.utils.apply(func, args, kwargs=None)>,
  <bound method Reduction.aggregate of <class 'dask.dataframe.dask_expr._reductions.Sum'>>,
  [[('chunk-066cb5e1c967740c2610f0628c801480', 0),
    ('chunk-066cb5e1c967740c2610f0628c801480', 1),
    ('chunk-066cb5e1c967740c2610f0628c801480', 2)]],
  {'skipna': True, 'axis': 0}),
 ('chunk-066cb5e1c967740c2610f0628c801480',
  0): <Task ('chunk-066cb5e1c967740c2610f0628c801480', 0) chunk(..., ...)>,
 ('chunk-066cb5e1c967740c2610f0628c801480',
  1): <Task ('chunk-066cb5e1c967740c2610f0628c801480', 1) chunk(..., ...)>,
 ('chunk-066cb5e1c967740c2610f0628c801480',
  2): <Task ('chunk-066cb5e1c967740c2610f0628c801480', 2) chunk(..., ...)>,
 ('getitem-474a30ec38b58de13c521b79814e62b8',
  0): <Task ('getitem-474a30ec38b58de13c521b79814e62b8', 0) getitem(...)>,
 ('getitem-474a30ec38b5